# Forecast Visualization — Model Evaluation & 7-Day Forecast (模型评估与7天预测可视化)

Interactive (Plotly) **head-to-head comparison** of the two NEWEST gradient-boosting models for
Finland electricity-price prediction — this is the project's goal #2 (compare XGBoost vs LightGBM):

| Algorithm | Newest model | Reported test MAE |
| --- | --- | --- |
| LightGBM | `lightgbm_v3_1` | 2.6390 |
| XGBoost | `xgboost_v4` | 2.7020 |

Sections:
1. **Head-to-head model evaluation** — both models on the SAME held-out test set
   (last 20%, chronological): metrics, actual-vs-predicted, residuals, feature importance.
2. **7-day forecast** — the latest predictions from `predictions/*.csv`.

The evaluation data is the shared **V3.1** feature table (V2.5 + grid `fi_*` + nuclear `nuclear_*`);
each model selects its own trained columns via its saved `feature_cols`.

Every figure renders interactively **directly in this notebook** — no image files are saved.

> Usage: run top to bottom. Re-run Section 2 after a new daily forecast
> (`python src/predict_system.py`) to refresh the charts.


## 0. Setup 


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ── paths (this notebook lives in data_visualization/) ───────────────────
DATA_PATH   = Path('../data/convertData/V3.1_15min_features.csv')
MODELS_DIR  = Path('../models/saved')
PREDICTIONS = Path('../predictions')

HELSINKI = 'Europe/Helsinki'


def show(fig):
    """Display a figure interactively in the notebook."""
    fig.show()


## 1. Model Evaluation — LightGBM V3.1 vs XGBoost V4 

Same data, same chronological 80/20 split, same metric definition — the ONLY difference
between the two is the algorithm. This is a **fair head-to-head comparison**.


In [3]:
# Load the V3.1 feature table and normalise the time axis
df = pd.read_csv(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert(HELSINKI)
df = df.sort_values('datetime').reset_index(drop=True)

# Chronological 80/20 split (no shuffle — time-series)
X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
train_end = n - int(n * 0.20)
X_test  = X.iloc[train_end:].reset_index(drop=True)
y_test  = y.iloc[train_end:].reset_index(drop=True)
test_dt = df['datetime'].iloc[train_end:].reset_index(drop=True)
print(f'Train rows: {train_end:,} | Test rows: {n - train_end:,}')


Train rows: 84,173 | Test rows: 21,043


In [4]:
# Evaluate the two NEWEST models head-to-head on the SAME chronological test set
MODEL_NAMES = ['lightgbm_v3_1', 'xgboost_v4']   # newest LightGBM and newest XGBoost

results = {}       # name -> {'MAE','RMSE','R2'}
preds    = {}      # name -> y_pred (numpy array)
models   = {}      # name -> trained model object
feature_cols = {}  # name -> list of trained feature column names

for name in MODEL_NAMES:
    meta = joblib.load(MODELS_DIR / f'{name}.pkl')
    model = meta['model']
    cols = meta['feature_cols']
    yp = model.predict(X_test[cols])

    results[name] = {
        'MAE':  mean_absolute_error(y_test, yp),
        'RMSE': np.sqrt(mean_squared_error(y_test, yp)),
        'R2':   r2_score(y_test, yp),
    }
    preds[name] = yp
    models[name] = model
    feature_cols[name] = cols
    print(f'{name}: MAE={results[name]["MAE"]:.4f} | '
          f'RMSE={results[name]["RMSE"]:.4f} | R2={results[name]["R2"]:.4f}')

# Side-by-side comparison table
comp = pd.DataFrame(results).T
print('\nComparison table (MAE/RMSE lower = better, R2 higher = better):')
print(comp.round(4))


lightgbm_v3_1: MAE=2.6390 | RMSE=7.8957 | R2=0.9740
xgboost_v4: MAE=2.7020 | RMSE=8.0376 | R2=0.9730

Comparison table (MAE/RMSE lower = better, R2 higher = better):
                 MAE    RMSE     R2
lightgbm_v3_1  2.639  7.8957  0.974
xgboost_v4     2.702  8.0376  0.973


In [5]:
# 1.1 Metrics comparison - MAE / RMSE / R2 side by side
metric_order = ['MAE', 'RMSE', 'R2']
fig = go.Figure()
for name in MODEL_NAMES:
    fig.add_trace(go.Bar(name=name, x=metric_order,
                         y=[comp.loc[name, m] for m in metric_order],
                         text=[f'{comp.loc[name, m]:.4f}' for m in metric_order],
                         textposition='outside'))
fig.update_layout(title='1.1 Metrics: LightGBM V3.1 vs XGBoost V4',
                  yaxis_title='Value', barmode='group')
show(fig)


In [6]:
# 1.2 Actual vs Predicted - time series (first 7 days of the test set)
steps = 7 * 24 * 4
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dt[:steps], y=y_test[:steps], name='Actual', mode='lines',
                         line=dict(color='black', width=2)))
for name in MODEL_NAMES:
    fig.add_trace(go.Scatter(x=test_dt[:steps], y=preds[name][:steps],
                             name=name, mode='lines', line=dict(width=1.5)))
fig.update_layout(title='1.2 Actual vs Predicted - First 7 Days of Test (LGBM V3.1 vs XGB V4)',
                  xaxis_title='Time', yaxis_title='Price (EUR/MWh)')
show(fig)


In [7]:
# 1.3 Actual vs Predicted - scatter with identity line (one panel per model)
from plotly.subplots import make_subplots

rng = np.random.RandomState(42)
idx = rng.choice(len(y_test), size=min(20000, len(y_test)), replace=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=MODEL_NAMES)
for c, name in enumerate(MODEL_NAMES, start=1):
    yt = y_test.values[idx]
    yp = preds[name][idx]
    fig.add_trace(go.Scatter(x=yt, y=yp, mode='markers', opacity=0.3,
                             marker=dict(size=3)), row=1, col=c)
    lo = float(min(yt.min(), yp.min()))
    hi = float(max(yt.max(), yp.max()))
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines', name='y=x',
                             line=dict(color='red', dash='dash')), row=1, col=c)
fig.update_layout(title='1.3 Actual vs Predicted Scatter (sampled)', showlegend=False, height=500)
fig.update_xaxes(title_text='Actual (EUR/MWh)', row=1, col=1)
fig.update_xaxes(title_text='Actual (EUR/MWh)', row=1, col=2)
fig.update_yaxes(title_text='Predicted (EUR/MWh)', row=1, col=1)
fig.update_yaxes(title_text='Predicted (EUR/MWh)', row=1, col=2)
show(fig)


In [8]:
# 1.4 Residual distribution (residual = actual - predicted), both models overlaid
fig = go.Figure()
for name in MODEL_NAMES:
    res = y_test.values - preds[name]
    fig.add_trace(go.Histogram(x=res, nbinsx=80, name=name, opacity=0.6))
fig.add_vline(x=0, line_dash='dash', line_color='red')
fig.update_layout(title='1.4 Residual Distribution (LGBM V3.1 vs XGB V4)',
                  xaxis_title='Residual (EUR/MWh)', yaxis_title='Count',
                  barmode='overlay')
show(fig)
for name in MODEL_NAMES:
    res = y_test.values - preds[name]
    print(f'{name}: residual mean={res.mean():.4f} std={res.std():.4f}')


lightgbm_v3_1: residual mean=0.0762 std=7.8953
xgboost_v4: residual mean=0.0775 std=8.0373


In [9]:
# 1.5 Feature importance - top 20 of each model (two panels)
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=MODEL_NAMES)
for c, name in enumerate(MODEL_NAMES, start=1):
    imp = (pd.Series(models[name].feature_importances_, index=feature_cols[name])
           .sort_values().tail(20))
    fig.add_trace(go.Bar(x=imp.values, y=imp.index, orientation='h', name=name),
                  row=1, col=c)
fig.update_layout(title='1.5 Top 20 Feature Importances', showlegend=False, height=700)
fig.update_xaxes(title_text='Importance', row=1, col=1)
fig.update_xaxes(title_text='Importance', row=1, col=2)
show(fig)


## 2. 7-Day Forecast 


In [10]:
def load_forecasts():
    """Load every <model>_forecasts.csv into a dict keyed by model name."""
    frames = {}
    for path in sorted(PREDICTIONS.glob('*_forecasts.csv')):
        f = pd.read_csv(path, parse_dates=['run_date', 'target_datetime'])
        name = path.stem.replace('_forecasts', '')
        if f['target_datetime'].dt.tz is None:
            f['target_datetime'] = f['target_datetime'].dt.tz_localize('UTC').dt.tz_convert(HELSINKI)
        else:
            f['target_datetime'] = f['target_datetime'].dt.tz_convert(HELSINKI)
        frames[name] = f
    return frames

forecasts = load_forecasts()
print('Loaded forecast files:', len(forecasts))
for name, f in forecasts.items():
    print(f'  {name}: {len(f)} rows | latest run {f["run_date"].max()}')


Loaded forecast files: 12
  lightgbm_v2_5_2: 13440 rows | latest run 2026-09-05 16:49:21.419027+03:00
  lightgbm_v2_5: 20160 rows | latest run 2026-09-05 16:49:21.419027+03:00
  lightgbm_v2: 5040 rows | latest run 2026-09-05 16:49:21.419027+03:00
  lightgbm_v3_1: 7392 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v1_5: 20160 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v1: 5040 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v2_5_2: 16128 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v2_5_3: 13440 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v2_5: 20160 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v2: 5040 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v3: 7392 rows | latest run 2026-09-05 16:49:21.419027+03:00
  xgboost_v4: 7392 rows | latest run 2026-09-05 16:49:21.419027+03:00


In [11]:
# 2.1 Latest 7-day forecast for every model on one chart
fig = go.Figure()
for name, f in forecasts.items():
    f_latest = f[f['run_date'] == f['run_date'].max()]
    fig.add_trace(go.Scatter(x=f_latest['target_datetime'], y=f_latest['predicted_price'],
                             name=name, mode='lines'))
fig.update_layout(title='2.1 7-Day Forecast - All Models ',
                  xaxis_title='Time', yaxis_title='Predicted Price (EUR/MWh)')
show(fig)


In [12]:
# 2.2 Head-to-head: newest LightGBM vs newest XGBoost 7-day forecast, actuals overlaid
compare = ['lightgbm_v3_1', 'xgboost_v4']
fig = go.Figure()
for name in compare:
    if name not in forecasts:
        print(f'  (no forecast file yet: {name})')
        continue
    f_latest = forecasts[name][forecasts[name]['run_date'] == forecasts[name]['run_date'].max()]
    fig.add_trace(go.Scatter(x=f_latest['target_datetime'], y=f_latest['predicted_price'],
                             name=name, mode='lines', line=dict(width=2)))
# overlay backfilled actuals once available (from whichever model has them)
for name in compare:
    if name not in forecasts:
        continue
    f_latest = forecasts[name][forecasts[name]['run_date'] == forecasts[name]['run_date'].max()]
    actual = f_latest.dropna(subset=['actual_price'])
    if not actual.empty:
        fig.add_trace(go.Scatter(x=actual['target_datetime'], y=actual['actual_price'],
                                 name='Actual (backfilled)', mode='lines',
                                 line=dict(color='black', dash='dash', width=2)))
        break
fig.update_layout(title='2.2 Latest 7-Day Forecast: LightGBM V3.1 vs XGBoost V4',
                  xaxis_title='Time', yaxis_title='Price (EUR/MWh)')
show(fig)
